# F1 MultiCircuit Lap Time Model — Google Colab Runner

**Antes de começar:**
1. Vá em **Runtime → Change runtime type** e selecione **T4 GPU** (necessário para o LSTM).
2. Execute as células em ordem.

---

### Como o repositório é carregado?

Escolha **uma** das duas opções na célula abaixo:

| Opção | Como funciona | Vantagem |
|---|---|---|
| `git` | Clona o repo direto do GitHub para `/content/` | Mais rápido; sem necessidade de Drive |
| `drive` | Usa uma cópia já salva no Google Drive | Resultados persistem entre sessões |

> **Atenção (opção `git`):** Os resultados ficam em `/content/` e são **perdidos** ao fechar a sessão. Use a célula de backup ao final para salvar no Drive ou baixar.

## 1. Configuração

In [ ]:
# @title Configuração geral { run: "auto" }

REPO_SOURCE = "git"  # @param ["git", "drive"]

# --- Opção git ---
# Para repositório PRIVADO: gere um token em https://github.com/settings/tokens
# e cole abaixo. Para repositório PÚBLICO, deixe GITHUB_TOKEN em branco.
GITHUB_REPO  = "emipe09/F1-MultiCircuit-LapTimeModel"  # @param {type:"string"}
GITHUB_TOKEN = ""  # @param {type:"string"}

# --- Opção drive ---
# Path do repo dentro do Google Drive
DRIVE_REPO_PATH = "/content/drive/MyDrive/TCC/F1-MultiCircuit-LapTimeModel"  # @param {type:"string"}

In [ ]:
import os, sys

if REPO_SOURCE == "git":
    REPO_PATH = "/content/F1-MultiCircuit-LapTimeModel"
    if os.path.isdir(REPO_PATH):
        print("Repositório já clonado. Atualizando...")
        !git -C {REPO_PATH} pull
    else:
        if GITHUB_TOKEN:
            clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        else:
            clone_url = f"https://github.com/{GITHUB_REPO}.git"
        !git clone {clone_url} {REPO_PATH}

elif REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_PATH = DRIVE_REPO_PATH
    assert os.path.isdir(REPO_PATH), f"Repositório não encontrado em: {REPO_PATH}"

os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "Scripts", "Source"))
print(f"\nDiretório de trabalho: {os.getcwd()}")

## 2. Instalar dependências

In [ ]:
!pip install -q -r Utils/requirements.txt
print("Dependências instaladas.")

## 3. Verificar GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU disponível: {gpus[0].name}")
else:
    print("AVISO: Nenhuma GPU detectada. O LSTM vai rodar em CPU (muito mais lento).")
    print("Para usar GPU: Runtime → Change runtime type → T4 GPU")

## 4. Rodar experimentos

| `MODELS` | Descrição |
|---|---|
| `lstm` | LSTM expanding-window |
| `xgb_ew` | XGBoost expanding-window |
| `lr_ew` | Linear Regression expanding-window |
| `xgb` | XGBoost sliding-window |
| `lr` | Linear Regression sliding-window |

> **Dica:** Após a primeira execução do LSTM, defina `use_saved_lstm_params: true` no YAML do circuito para reutilizar os params do Optuna e poupar tempo nas execuções seguintes.

In [ ]:
# @title Parâmetros do experimento { run: "auto" }
GP     = "bahrain"  # @param ["bahrain", "saudi", "usa", "italy", "hungary", "all"]
MODELS = "lstm"     # @param ["lstm", "lr_ew", "xgb_ew", "lr", "xgb", "lr_ew xgb_ew lstm", "lr xgb"]

In [ ]:
gp_flag = "" if GP == "all" else f"--configs {GP}.yaml"
cmd = f"python Scripts/Source/run_all_models.py {gp_flag} --models {MODELS}"
print(f"Executando: {cmd}\n")
!{cmd}

## 5. Inspecionar resultados

In [ ]:
import json, pandas as pd
from pathlib import Path

safe_name = GP.lower().replace(" ", "_")

lstm_meta_path = Path(f"Scripts/Results/lstm/ew/models/{safe_name}_lstm_model_ew_metadata.json")
if lstm_meta_path.exists():
    meta = json.loads(lstm_meta_path.read_text())
    print("=== LSTM EW — Métricas de validação ===")
    for k, v in meta.get("val_metrics", {}).items():
        print(f"  {k}: {v:.4f}")
    print(f"\nSequence length : {meta.get('sequence_length')}")
    print(f"Final epochs    : {meta.get('final_epoch_count')}")
    if meta.get("optuna_summary"):
        print(f"Optuna best RMSE: {meta['optuna_summary'].get('best_value', 'N/A'):.4f}")
        print(f"Optuna best params: {meta['optuna_summary'].get('best_params')}")
else:
    print(f"Metadata não encontrado em: {lstm_meta_path}")

trials_path = Path(f"Scripts/Results/lstm/ew/params/{safe_name}_lstm_optuna_trials_ew.csv")
if trials_path.exists():
    trials_df = pd.read_csv(trials_path)
    print(f"\n=== Optuna trials ({len(trials_df)} total) — top 5 por RMSE ===")
    print(trials_df.sort_values("rmse").head(5).to_string(index=False))

## 6. Salvar resultados (apenas para opção `git`)

Como `/content/` é volátil, salve os resultados antes de fechar a sessão.
Escolha uma das opções abaixo.

In [ ]:
# Opção A: copiar para o Google Drive
if REPO_SOURCE == "git":
    from google.colab import drive as _drive
    _drive.mount("/content/drive")

DRIVE_RESULTS = "/content/drive/MyDrive/TCC/F1-Results"
!mkdir -p {DRIVE_RESULTS}
!cp -r Scripts/Results/. {DRIVE_RESULTS}/
print(f"Resultados copiados para: {DRIVE_RESULTS}")

In [ ]:
# Opção B: baixar como ZIP
from google.colab import files
!zip -r /content/results.zip Scripts/Results/
files.download("/content/results.zip")

## 7. (Opcional) MLflow UI via ngrok

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, time

proc = subprocess.Popen(["mlflow", "ui", "--backend-store-uri", "Scripts/Results/mlruns", "--port", "5000"])
time.sleep(3)
tunnel = ngrok.connect(5000)
print(f"MLflow UI: {tunnel.public_url}")